# Clase de Data Science · Procesamiento de texto en Python  
**Sesión:** Introducción al procesamiento de textos y aprendizaje automático para texto  
**Duración:** 100 minutos · **Nivel:** Principiante

> En esta clase aprenderás a transformar textos en datos utilizables para análisis y modelos sencillos de Machine Learning.


## Fecha
> Completa aquí la fecha de la sesión.


## Objetivo de la sesión (100 min)

Al finalizar esta clase, podrás:

1. Explicar qué es el **procesamiento de texto** y por qué es importante en Data Science.  
2. Reconocer las etapas básicas de un flujo de trabajo de **Machine Learning para texto**.  
3. Limpiar y normalizar textos en Python usando operaciones simples y **expresiones regulares**.  
4. Convertir documentos en números con **Bolsa de Palabras (BoW)** y **TF-IDF**.  
5. Entrenar un modelo base de clasificación de sentimiento usando un dataset sintético.  
6. Interpretar por qué un modelo de texto necesita una buena representación antes de aprender.


## Agenda sugerida (100 minutos)

1. Contexto de la clase + mapa general de NLP para principiantes (10 min)  
2. **Actividad 0 (breakout rooms):** leer textos como analista de datos (10 min)  
3. Generación y exploración del dataset sintético (10 min)  
4. Ejercicio 1: inspección inicial del dataset y longitud de textos (15 min)  
5. Ejercicio 2: limpieza y normalización con Python + regex (15 min)  
6. Ejercicio 3: tokenización y stopwords básicas (15 min)  
7. Ejercicio 4: vectorización con Bolsa de Palabras (10 min)  
8. Ejercicio 5: TF-IDF y términos relevantes por documento (10 min)  
9. Ejercicio 6: mini modelo de clasificación de sentimiento (15 min)


## Contexto de la clase

El texto está en todas partes: reseñas de clientes, tickets de soporte, comentarios, chats, encuestas, noticias, redes sociales y documentos internos.  
El problema es que el texto **no viene listo para un modelo**. Antes de modelar, debemos convertirlo en una forma que la computadora pueda interpretar.

En una primera aproximación, el flujo suele verse así:

1. **Recolectar textos** y, si existe, la etiqueta objetivo.  
2. **Limpiar y normalizar**: minúsculas, quitar ruido, URLs, signos innecesarios, etc.  
3. **Tokenizar**: separar el texto en unidades pequeñas, normalmente palabras.  
4. **Representar** el texto con números: BoW, n-gramas, TF-IDF o embeddings.  
5. **Entrenar** un modelo con esas representaciones numéricas.  
6. **Evaluar** si el modelo está aprendiendo algo útil.

En esta clase haremos una versión **introductoria y muy guiada** de ese flujo.


## Introducción: ¿qué es procesar texto?

**Procesamiento de texto** significa convertir lenguaje natural en una estructura que podamos analizar con Python.

### Ideas clave para esta sesión

- Un texto es una secuencia de caracteres; para analizarlo, primero hay que **prepararlo**.
- No todo lo que aparece en un texto aporta valor: URLs, menciones, símbolos repetidos y ruido visual pueden estorbar.
- Un modelo de Machine Learning **no entiende palabras directamente**; necesita matrices numéricas.
- Dos técnicas base muy importantes son:
  - **Bolsa de Palabras (Bag of Words / BoW):** cuenta cuántas veces aparece cada término.
  - **TF-IDF:** da más peso a términos importantes dentro de un documento y menos a términos demasiado comunes.
- En proyectos reales también se usan **lemmatización**, **stemming** y **embeddings**; hoy los mencionaremos, pero trabajaremos con herramientas base para que el proceso sea claro.

### ¿En qué casos se usa?

- Análisis de sentimiento  
- Clasificación de tickets o correos  
- Detección de spam  
- Búsqueda y recuperación de información  
- Agrupación de comentarios similares  
- Sistemas de recomendación y soporte automatizado


## Dataset para la sesión

Usaremos un dataset **sintético** de reseñas cortas en español con dos etiquetas:

- `positivo`
- `negativo`

Además, el texto tendrá algo de **ruido intencional**: mayúsculas, signos repetidos, hashtags, URLs, menciones y pequeñas variaciones.  
Esto nos permite practicar una situación más parecida a la vida real.

Columnas principales:

- `review_id` — identificador del comentario  
- `texto` — reseña original  
- `sentimiento` — etiqueta objetivo  
- `canal` — origen ficticio del comentario  

> También guardaremos el dataset en un archivo local llamado `resenas_texto_sinteticas.csv`.


---

# Actividad 0 · Calentamiento (10 min)

### Instrucciones (breakout rooms, 3–4 personas)

Lean estos textos y conversen:

1. ¿Qué señales les hacen pensar que el sentimiento es positivo o negativo?  
2. ¿Qué partes del texto parecen **ruido** y podrían limpiarse?  
3. ¿Qué creen que tendría que hacer Python antes de entrenar un modelo?

### Textos de ejemplo

- `ME encantó la app!!! llegó rápido :) #feliz`
- `La entrega fue terrible, no lo recomiendo @soporte`
- `Excelente atención, volvería a comprar 100%`
- `Pésimo servicio... la app falla cada rato https://soporte.fake`

### Cierre esperado

Al final de la conversación, el grupo debería concluir algo como esto:

- El modelo necesita palabras o patrones que ayuden a diferenciar clases.
- Antes de modelar conviene limpiar ruido.
- Hay que convertir los textos en una **representación numérica**.


In [5]:
# ============================================================
# Imports y configuración (ejecuta esta celda primero)
# ============================================================
import re
import random
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

random.seed(42)
np.random.seed(42)


### Crear dataset sintético (para la práctica)

> Ejecuta esta celda para crear `df` y un archivo local `resenas_texto_sinteticas.csv`.


In [6]:
# ============================================================
# Dataset sintético de reseñas cortas
# ============================================================
positivos = [
    "Me encantó el producto, funciona perfecto",
    "Excelente calidad y entrega rápida",
    "Muy satisfecho con la compra, lo recomiendo",
    "El soporte fue amable y resolvió todo",
    "La app es fácil de usar y muy útil",
    "La comida estuvo deliciosa y fresca",
    "Buen servicio y atención muy rápida",
    "Todo llegó en buen estado y a tiempo",
    "La experiencia fue excelente de principio a fin",
    "Volvería a comprar sin pensarlo",
    "El curso estuvo claro y bien explicado",
    "Me gustó mucho, superó mis expectativas",
]

negativos = [
    "No me gustó el producto, llegó dañado",
    "Pésima calidad y demora en la entrega",
    "Muy decepcionado con la compra",
    "El soporte no respondió mi problema",
    "La app falla y se cierra sola",
    "La comida llegó fría y sin sabor",
    "Mal servicio y atención muy lenta",
    "El pedido llegó tarde y en mal estado",
    "La experiencia fue frustrante de principio a fin",
    "No volvería a comprar",
    "El curso estuvo confuso y mal explicado",
    "Esperaba mucho más, fue una mala experiencia",
]

ruidos = [
    "!!!", "...", " :)", " :(", " #feliz", " #enojo",
    " @marca", " @soporte", " visita https://ejemplo.com",
    " 100%", " en serio", " por favor"
]

canales = ["web", "app", "encuesta", "chat"]

def agregar_ruido(texto):
    extra1 = random.choice(ruidos)
    extra2 = random.choice(ruidos)
    nuevo = texto + extra1 + extra2
    
    if random.random() < 0.25:
        nuevo = nuevo.upper()
    elif random.random() < 0.25:
        nuevo = nuevo.capitalize()
        
    return nuevo

registros = []

review_id = 1001
for texto in positivos:
    for _ in range(3):
        registros.append({
            "review_id": review_id,
            "texto": agregar_ruido(texto),
            "sentimiento": "positivo",
            "canal": random.choice(canales)
        })
        review_id += 1

for texto in negativos:
    for _ in range(3):
        registros.append({
            "review_id": review_id,
            "texto": agregar_ruido(texto),
            "sentimiento": "negativo",
            "canal": random.choice(canales)
        })
        review_id += 1

df = pd.DataFrame(registros)

# Mezclar filas para que positivos y negativos no queden agrupados
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Guardar CSV local
df.to_csv("resenas_texto_sinteticas.csv", index=False)

print("Archivo guardado:", "resenas_texto_sinteticas.csv")
print("Dimensiones del dataset:", df.shape)


Archivo guardado: resenas_texto_sinteticas.csv
Dimensiones del dataset: (72, 4)


In [7]:
# Vista rápida del dataset
df.head(10)


,review_id,texto,sentimiento,canal
0,1005,EXCELENTE CALIDAD Y ENTREGA RÁPIDA VISITA HTTP...,positivo,app
1,1063,LA EXPERIENCIA FUE FRUSTRANTE DE PRINCIPIO A F...,negativo,encuesta
2,1019,Buen servicio y atención muy rápida!!! #enojo,positivo,encuesta
3,1001,"ME ENCANTÓ EL PRODUCTO, FUNCIONA PERFECTO EN S...",positivo,encuesta
4,1029,VOLVERÍA A COMPRAR SIN PENSARLO :) @SOPORTE,positivo,encuesta
5,1051,La app falla y se cierra sola :( :),negativo,chat
6,1011,El soporte fue amable y resolvió todo @marca...,positivo,encuesta
7,1035,"Me gustó mucho, superó mis expectativas... por...",positivo,app
8,1013,La app es fácil de usar y muy útil #feliz...,positivo,chat
9,1055,MAL SERVICIO Y ATENCIÓN MUY LENTA POR FAVOR #E...,negativo,chat


---

# Ejercicio 1 · Primer vistazo al dataset (15 min)

## Fundamentación teórica

Antes de limpiar o modelar texto, conviene responder preguntas básicas:

- ¿Cuántos registros tengo?
- ¿Cuáles son las columnas?
- ¿Las clases están balanceadas?
- ¿Los textos son muy largos o muy cortos?

Este paso es importante porque un modelo de texto puede comportarse distinto si:
- hay clases desbalanceadas,
- los textos son extremadamente cortos,
- hay columnas mal definidas,
- o el dataset es demasiado pequeño.

## Objetivo

Explorar el dataset y calcular medidas simples de longitud de texto para entender con qué tipo de datos vamos a trabajar.


### Tu turno

Escribe código para hacer lo siguiente:

1. Ver la forma del dataset.  
2. Revisar nombres de columnas.  
3. Contar cuántos textos hay por clase (`positivo` / `negativo`).  
4. Calcular:
   - número de palabras por texto,
   - número de caracteres por texto.  
5. Obtener un resumen descriptivo de esas longitudes.


In [8]:
# Escribe tu solución aquí.
# Sugerencias:
# - usa df.shape
# - usa df.columns
# - usa value_counts()
# - usa str.split().apply(len)
# - usa str.len()

df.head(3)


,review_id,texto,sentimiento,canal
0,1005,EXCELENTE CALIDAD Y ENTREGA RÁPIDA VISITA HTTP...,positivo,app
1,1063,LA EXPERIENCIA FUE FRUSTRANTE DE PRINCIPIO A F...,negativo,encuesta
2,1019,Buen servicio y atención muy rápida!!! #enojo,positivo,encuesta


### Pista

- `df["sentimiento"].value_counts()` te ayuda a revisar balance de clases.  
- `df["texto"].str.split().apply(len)` cuenta palabras.  
- `df["texto"].str.len()` cuenta caracteres.


In [26]:
# ✅ Solución
print("Forma del dataset:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

print("\nConteo por clase:")
print(df["sentimiento"].value_counts())


Forma del dataset: (72, 8)

Columnas:
['review_id', 'texto', 'sentimiento', 'canal', 'num_palabras', 'num_caracteres', 'texto_limpio', 'tokens']

Conteo por clase:
sentimiento
positivo    36
negativo    36
Name: count, dtype: int64


In [27]:
df["num_palabras"] = df["texto"].str.split().apply(len)
df["num_caracteres"] = df["texto"].str.len()

In [28]:
print("\nResumen descriptivo de longitudes:")
display(df[["num_palabras", "num_caracteres"]].describe())


Resumen descriptivo de longitudes:


,num_palabras,num_caracteres
count,72.000000,72.000000
mean,8.805556,51.680556
std,1.349375,12.368550
min,5.000000,31.000000
25%,8.000000,44.750000
50%,9.000000,49.000000
75%,10.000000,55.000000
max,12.000000,92.000000


In [29]:
print("\nEjemplos de textos:")
display(df[["texto", "sentimiento", "num_palabras", "num_caracteres"]].head(5))


Ejemplos de textos:


,texto,sentimiento,num_palabras,num_caracteres
0,EXCELENTE CALIDAD Y ENTREGA RÁPIDA VISITA HTTP...,positivo,8,66
1,LA EXPERIENCIA FUE FRUSTRANTE DE PRINCIPIO A F...,negativo,10,78
2,Buen servicio y atención muy rápida!!! #enojo,positivo,7,45
3,"ME ENCANTÓ EL PRODUCTO, FUNCIONA PERFECTO EN S...",positivo,8,53
4,VOLVERÍA A COMPRAR SIN PENSARLO :) @SOPORTE,positivo,7,43


---

# Ejercicio 2 · Limpieza y normalización de texto (15 min)

## Fundamentación teórica

Los textos reales suelen traer elementos que dificultan el análisis:

- mayúsculas y minúsculas mezcladas,
- tildes,
- URLs,
- menciones,
- hashtags,
- signos repetidos,
- números irrelevantes,
- espacios extra.

La **limpieza** busca estandarizar el texto.  
No existe una única limpieza correcta: depende del problema.  
Por ejemplo, en algunos casos podrías querer conservar emojis o números.  
Aquí haremos una limpieza base pensada para una primera aproximación a clasificación de sentimiento.

## Objetivo

Crear una función `limpiar_texto()` que normalice los comentarios para que tengan una forma más consistente antes de tokenizar y vectorizar.


### Tu turno

Crea una función que haga, al menos, estos pasos:

1. Convertir a minúsculas.  
2. Quitar tildes.  
3. Eliminar URLs.  
4. Eliminar menciones (`@usuario`) y hashtags (`#tema`).  
5. Eliminar signos de puntuación y números.  
6. Quitar espacios duplicados.

Luego aplícala a una nueva columna llamada `texto_limpio`.


In [10]:
# Escribe aquí tu versión inicial.
# Puedes reemplazar esta función por tu propia solución.

def limpiar_texto(texto):
    texto = str(texto).lower()
    return texto

df["texto_limpio"] = df["texto"].apply(limpiar_texto)
df[["texto", "texto_limpio"]].head(3)


,texto,texto_limpio
0,EXCELENTE CALIDAD Y ENTREGA RÁPIDA VISITA HTTP...,excelente calidad y entrega rápida visita http...
1,LA EXPERIENCIA FUE FRUSTRANTE DE PRINCIPIO A F...,la experiencia fue frustrante de principio a f...
2,Buen servicio y atención muy rápida!!! #enojo,buen servicio y atención muy rápida!!! #enojo


### Pista

- Para quitar tildes puedes usar `unicodedata.normalize`.  
- Para URLs: `re.sub(r"http\S+|www\.\S+", " ", texto)`  
- Para menciones: `re.sub(r"@\w+", " ", texto)`  
- Para hashtags: `re.sub(r"#\w+", " ", texto)`  
- Para espacios repetidos: `re.sub(r"\s+", " ", texto).strip()`


In [11]:
# ✅ Solución
def quitar_tildes(texto):
    return "".join(
        caracter
        for caracter in unicodedata.normalize("NFD", texto)
        if unicodedata.category(caracter) != "Mn"
    )

def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = quitar_tildes(texto)
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)
    texto = re.sub(r"@\w+", " ", texto)
    texto = re.sub(r"#\w+", " ", texto)
    texto = re.sub(r"[^a-z\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df["texto_limpio"] = df["texto"].apply(limpiar_texto)

display(df[["texto", "texto_limpio"]].head(8))


,texto,texto_limpio
0,EXCELENTE CALIDAD Y ENTREGA RÁPIDA VISITA HTTP...,excelente calidad y entrega rapida visita
1,LA EXPERIENCIA FUE FRUSTRANTE DE PRINCIPIO A F...,la experiencia fue frustrante de principio a f...
2,Buen servicio y atención muy rápida!!! #enojo,buen servicio y atencion muy rapida
3,"ME ENCANTÓ EL PRODUCTO, FUNCIONA PERFECTO EN S...",me encanto el producto funciona perfecto en serio
4,VOLVERÍA A COMPRAR SIN PENSARLO :) @SOPORTE,volveria a comprar sin pensarlo
5,La app falla y se cierra sola :( :),la app falla y se cierra sola
6,El soporte fue amable y resolvió todo @marca...,el soporte fue amable y resolvio todo
7,"Me gustó mucho, superó mis expectativas... por...",me gusto mucho supero mis expectativas por favor


In [12]:
# Comparación rápida de un ejemplo
fila = 0
print("Texto original:")
print(df.loc[fila, "texto"])

print("\nTexto limpio:")
print(df.loc[fila, "texto_limpio"])


Texto original:
EXCELENTE CALIDAD Y ENTREGA RÁPIDA VISITA HTTPS://EJEMPLO.COM 100%

Texto limpio:
excelente calidad y entrega rapida visita


---

# Ejercicio 3 · Tokenización y stopwords básicas (15 min)

## Fundamentación teórica

**Tokenizar** significa dividir un texto en piezas más pequeñas, normalmente palabras.  
Es uno de los pasos centrales del procesamiento de texto porque muchos métodos trabajan sobre tokens.

Además, suele ser útil remover algunas palabras muy frecuentes llamadas **stopwords**:
- artículos,
- preposiciones,
- conectores,
- palabras muy generales.

Ojo: quitar stopwords no siempre mejora el resultado; depende del problema.  
En esta clase usaremos una lista pequeña y manual para entender la idea.

## Objetivo

Construir una función `tokenizar()` que reciba un texto, lo limpie y devuelva una lista de tokens filtrados con stopwords básicas.


### Tu turno

1. Crea una lista o conjunto de stopwords simples en español.  
2. Construye una función `tokenizar(texto)` que:
   - use `limpiar_texto(texto)`,
   - separe por espacios,
   - elimine stopwords,
   - elimine tokens de longitud 1.  
3. Aplica la función a una columna llamada `tokens`.


In [13]:
# Escribe aquí tu solución.

stopwords_basicas = {
    "el", "la", "los", "las", "de", "del", "y", "o", "a", "en",
    "un", "una", "muy", "con", "mi", "me"
}

def tokenizar(texto):
    texto = limpiar_texto(texto)
    tokens = texto.split()
    return tokens

df["tokens"] = df["texto"].apply(tokenizar)
df[["texto_limpio", "tokens"]].head(3)


,texto_limpio,tokens
0,excelente calidad y entrega rapida visita,"[excelente, calidad, y, entrega, rapida, visita]"
1,la experiencia fue frustrante de principio a f...,"[la, experiencia, fue, frustrante, de, princip..."
2,buen servicio y atencion muy rapida,"[buen, servicio, y, atencion, muy, rapida]"


### Pista

- Un `set` es muy útil para buscar rápidamente si una palabra está en la lista de stopwords.  
- Puedes filtrar así:  
  `tokens_filtrados = [t for t in tokens if t not in stopwords_basicas and len(t) > 1]`


In [14]:
# ✅ Solución
stopwords_basicas = {
    "el", "la", "los", "las", "de", "del", "y", "o", "a", "en",
    "un", "una", "muy", "con", "mi", "me", "lo", "al", "por",
    "para", "que", "se", "sin", "fue", "es", "todo"
}

def tokenizar(texto):
    texto = limpiar_texto(texto)
    tokens = texto.split()
    tokens_filtrados = [
        token for token in tokens
        if token not in stopwords_basicas and len(token) > 1
    ]
    return tokens_filtrados

df["tokens"] = df["texto"].apply(tokenizar)

display(df[["texto_limpio", "tokens"]].head(8))


,texto_limpio,tokens
0,excelente calidad y entrega rapida visita,"[excelente, calidad, entrega, rapida, visita]"
1,la experiencia fue frustrante de principio a f...,"[experiencia, frustrante, principio, fin, visita]"
2,buen servicio y atencion muy rapida,"[buen, servicio, atencion, rapida]"
3,me encanto el producto funciona perfecto en serio,"[encanto, producto, funciona, perfecto, serio]"
4,volveria a comprar sin pensarlo,"[volveria, comprar, pensarlo]"
5,la app falla y se cierra sola,"[app, falla, cierra, sola]"
6,el soporte fue amable y resolvio todo,"[soporte, amable, resolvio]"
7,me gusto mucho supero mis expectativas por favor,"[gusto, mucho, supero, mis, expectativas, favor]"


In [15]:
# Contar palabras más frecuentes después de tokenizar
todos_los_tokens = [token for lista in df["tokens"] for token in lista]
frecuencias_tokens = Counter(todos_los_tokens)

top_10 = pd.DataFrame(
    frecuencias_tokens.most_common(10),
    columns=["token", "frecuencia"]
)

top_10


,token,frecuencia
0,serio,18
1,visita,13
2,llego,12
3,favor,11
4,experiencia,9
5,mal,9
6,no,9
7,estuvo,9
8,excelente,6
9,calidad,6


---

# Ejercicio 4 · Vectorización con Bolsa de Palabras (BoW) (10 min)

## Fundamentación teórica

Los modelos de Machine Learning trabajan con números, no con frases completas.  
Por eso necesitamos una forma de convertir cada documento en un vector numérico.

La **Bolsa de Palabras (BoW)** construye una matriz donde:

- cada fila es un documento,
- cada columna es una palabra o n-grama,
- cada celda indica cuántas veces aparece ese término.

### Idea clave

BoW **ignora el orden global** del texto y se concentra en la presencia o frecuencia de términos.  
Aun así, es una técnica muy útil como línea base.

## Objetivo

Crear una matriz BoW con `CountVectorizer` y observar:

- el tamaño de la matriz,
- parte del vocabulario,
- los términos más frecuentes del corpus.


### Tu turno

1. Crea un `CountVectorizer`.  
2. Usa `tokenizar` como función de tokenización.  
3. Prueba con `ngram_range=(1,2)` para incluir unigramas y bigramas.  
4. Ajusta el vectorizador con `df["texto"]`.  
5. Revisa el `shape` de la matriz.


In [30]:
# Escribe aquí tu solución.
# Recuerda: si usas tokenizer personalizado en CountVectorizer,
# es buena idea fijar token_pattern=None.

vectorizador_bow = CountVectorizer()
X_bow = vectorizador_bow.fit_transform(df["texto"])

print("Shape de la matriz BoW:", X_bow.shape)


Shape de la matriz BoW: (72, 90)


### Pista

- `CountVectorizer(tokenizer=tokenizar, token_pattern=None, ngram_range=(1,2))`  
- El vocabulario se puede ver con `get_feature_names_out()`.  
- La frecuencia total por término puede obtenerse sumando columnas de la matriz.


In [17]:
# ✅ Solución
vectorizador_bow = CountVectorizer(
    tokenizer=tokenizar,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=1
)

X_bow = vectorizador_bow.fit_transform(df["texto"])

print("Shape de la matriz BoW:", X_bow.shape)

vocabulario = vectorizador_bow.get_feature_names_out()
print("\nPrimeros 20 términos del vocabulario:")
print(vocabulario[:20])


Shape de la matriz BoW: (72, 168)

Primeros 20 términos del vocabulario:
['amable' 'amable resolvio' 'app' 'app facil' 'app falla' 'atencion'
 'atencion lenta' 'atencion rapida' 'bien' 'bien explicado' 'buen'
 'buen estado' 'buen servicio' 'calidad' 'calidad demora'
 'calidad entrega' 'cierra' 'cierra sola' 'claro' 'claro bien']


In [18]:
# Términos más frecuentes del corpus
frecuencias_bow = np.asarray(X_bow.sum(axis=0)).ravel()
terminos_bow = vectorizador_bow.get_feature_names_out()

top_idx = frecuencias_bow.argsort()[::-1][:15]

top_bow = pd.DataFrame({
    "termino": terminos_bow[top_idx],
    "frecuencia": frecuencias_bow[top_idx]
})

top_bow


,termino,frecuencia
0,serio,18
1,visita,13
2,llego,12
3,favor,11
4,no,9
5,mal,9
6,experiencia,9
7,estuvo,9
8,producto,6
9,principio fin,6


---

# Ejercicio 5 · TF-IDF: términos importantes por documento (10 min)

## Fundamentación teórica

BoW cuenta apariciones, pero no distingue muy bien entre palabras comunes y palabras realmente informativas.

**TF-IDF** combina dos ideas:

- **TF (Term Frequency):** qué tan frecuente es un término dentro de un documento.
- **IDF (Inverse Document Frequency):** penaliza términos que aparecen en demasiados documentos.

### Intuición

Si una palabra aparece en casi todos los textos, suele aportar poco para diferenciar documentos.  
Si aparece mucho en un documento pero poco en el resto, probablemente es más informativa.

## Objetivo

Construir una representación TF-IDF y revisar qué palabras o n-gramas reciben mayor peso en un texto específico.


### Tu turno

1. Crea un `TfidfVectorizer`.  
2. Usa la misma tokenización del ejercicio anterior.  
3. Ajusta el vectorizador con todos los textos.  
4. Escoge un documento y revisa sus términos con mayor peso.


In [19]:
# Escribe aquí tu solución.
vectorizador_tfidf = TfidfVectorizer()
X_tfidf = vectorizador_tfidf.fit_transform(df["texto"])

print("Shape TF-IDF:", X_tfidf.shape)


Shape TF-IDF: (72, 90)


### Pista

- `row = X_tfidf[doc_id].toarray().ravel()` convierte un documento a un vector denso.  
- `argsort()[::-1]` ayuda a ordenar de mayor a menor.  
- Usa `get_feature_names_out()` para recuperar los nombres de los términos.


In [20]:
# ✅ Solución
vectorizador_tfidf = TfidfVectorizer(
    tokenizer=tokenizar,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=1
)

X_tfidf = vectorizador_tfidf.fit_transform(df["texto"])

print("Shape TF-IDF:", X_tfidf.shape)


Shape TF-IDF: (72, 168)


In [21]:
# Términos con mayor peso TF-IDF para un documento ejemplo
doc_id = 0
fila = X_tfidf[doc_id].toarray().ravel()
terminos_tfidf = vectorizador_tfidf.get_feature_names_out()

top_idx = fila.argsort()[::-1][:10]

print("Documento original:")
print(df.loc[doc_id, "texto"])
print("\nDocumento limpio:")
print(df.loc[doc_id, "texto_limpio"])
print("\nTop términos por TF-IDF:")

for i in top_idx:
    if fila[i] > 0:
        print(f"{terminos_tfidf[i]} -> {fila[i]:.3f}")


Documento original:
EXCELENTE CALIDAD Y ENTREGA RÁPIDA VISITA HTTPS://EJEMPLO.COM 100%

Documento limpio:
excelente calidad y entrega rapida visita

Top términos por TF-IDF:
rapida visita -> 0.421
excelente calidad -> 0.357
calidad entrega -> 0.357
entrega rapida -> 0.357
entrega -> 0.306
rapida -> 0.306
calidad -> 0.306
excelente -> 0.306
visita -> 0.257


---

# Ejercicio 6 · Mini proyecto: clasificación de sentimiento con TF-IDF + Regresión Logística (15 min)

## Fundamentación teórica

Ya tenemos una representación numérica del texto.  
Ahora podemos usarla como entrada para un modelo supervisado.

En esta clase construiremos un flujo simple:

1. Separar `train` y `test`.  
2. Ajustar TF-IDF solo con los datos de entrenamiento.  
3. Entrenar un clasificador (`LogisticRegression`).  
4. Predecir sobre test.  
5. Evaluar con métricas básicas.

### Importante

Este dataset es **sintético y pequeño**, así que el objetivo de la actividad no es obtener un sistema de producción.  
El objetivo es entender el **pipeline básico** de ML para texto.

## Objetivo

Entrenar un modelo sencillo de análisis de sentimiento y revisar si logra diferenciar textos positivos y negativos.


### Tu turno

Completa los pasos del pipeline:

1. Separa `X` e `y`.  
2. Haz `train_test_split`.  
3. Vectoriza con `TfidfVectorizer`.  
4. Entrena `LogisticRegression`.  
5. Calcula `accuracy` y revisa la matriz de confusión.


In [31]:
# Escribe aquí tu solución.
X = df["texto"]
y = df["sentimiento"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Train:", len(X_train))
print("Test:", len(X_test))


Train: 54
Test: 18


In [32]:
X

0     EXCELENTE CALIDAD Y ENTREGA RÁPIDA VISITA HTTP...
1     LA EXPERIENCIA FUE FRUSTRANTE DE PRINCIPIO A F...
2         Buen servicio y atención muy rápida!!! #enojo
3     ME ENCANTÓ EL PRODUCTO, FUNCIONA PERFECTO EN S...
4           VOLVERÍA A COMPRAR SIN PENSARLO :) @SOPORTE
                            ...                        
67           Todo llegó en buen estado y a tiempo... :)
68        BUEN SERVICIO Y ATENCIÓN MUY RÁPIDA :) #FELIZ
69    La experiencia fue frustrante de principio a f...
70         La app es fácil de usar y muy útil #enojo :(
71               La comida llegó fría y sin sabor!!! :)
Name: texto, Length: 72, dtype: object

In [33]:
y

0     positivo
1     negativo
2     positivo
3     positivo
4     positivo
        ...   
67    positivo
68    positivo
69    negativo
70    positivo
71    negativo
Name: sentimiento, Length: 72, dtype: object

### Pista

- Usa `stratify=y` dentro de `train_test_split` para conservar proporciones por clase.  
- Ajusta el vectorizador con `X_train` y transforma `X_test`.  
- `LogisticRegression(max_iter=1000)` suele evitar problemas de convergencia en ejemplos pequeños.  
- Evalúa con:
  - `accuracy_score`
  - `confusion_matrix`
  - `classification_report`


In [23]:
# ✅ Solución
X = df["texto"]
y = df["sentimiento"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

vectorizador_modelo = TfidfVectorizer(
    tokenizer=tokenizar,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=1
)

Xtr = vectorizador_modelo.fit_transform(X_train)
Xte = vectorizador_modelo.transform(X_test)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(Xtr, y_train)

predicciones = modelo.predict(Xte)

accuracy = accuracy_score(y_test, predicciones)
matriz = confusion_matrix(y_test, predicciones)
reporte = classification_report(y_test, predicciones)

print(f"Accuracy: {accuracy:.3f}")
print("\nMatriz de confusión:")
print(matriz)

print("\nReporte de clasificación:")
print(reporte)


Accuracy: 0.889

Matriz de confusión:
[[9 0]
 [2 7]]

Reporte de clasificación:
              precision    recall  f1-score   support

    negativo       0.82      1.00      0.90         9
    positivo       1.00      0.78      0.88         9

    accuracy                           0.89        18
   macro avg       0.91      0.89      0.89        18
weighted avg       0.91      0.89      0.89        18



In [24]:
# Probar el modelo con ejemplos nuevos
ejemplos_nuevos = [
    "Excelente servicio, llegó rápido y en buen estado",
    "La app se cierra todo el tiempo, pésima experiencia",
    "Buen soporte y atención clara",
    "No lo recomiendo, fue frustrante"
]

X_nuevos = vectorizador_modelo.transform(ejemplos_nuevos)
pred_nuevas = modelo.predict(X_nuevos)

pd.DataFrame({
    "texto": ejemplos_nuevos,
    "prediccion": pred_nuevas
})


,texto,prediccion
0,"Excelente servicio, llegó rápido y en buen estado",positivo
1,"La app se cierra todo el tiempo, pésima experi...",negativo
2,Buen soporte y atención clara,positivo
3,"No lo recomiendo, fue frustrante",negativo


---

# Concepto extra · ¿Qué son los embeddings? (explicación breve)

En esta clase usamos **BoW** y **TF-IDF**, que son representaciones muy valiosas para empezar.  
Pero en NLP moderno también se usan **embeddings**.

## Idea intuitiva

Un embedding representa palabras o textos como vectores densos en un espacio numérico donde:

- términos parecidos quedan cerca,
- términos diferentes quedan más lejos,
- parte de la relación semántica queda capturada en distancias.

### Ejemplos de familias de embeddings

- **Word2Vec**
- **fastText**
- **BERT / Sentence Transformers**

## Diferencia rápida frente a BoW

- **BoW / TF-IDF:** se enfocan en conteos o pesos por término.  
- **Embeddings:** buscan capturar similitud semántica y contexto de forma más rica.

Para una primera clase, empezar con BoW y TF-IDF suele ser mejor porque:
- son más fáciles de interpretar,
- exigen menos recursos,
- permiten entender el pipeline básico con claridad.


---

# ✅ Solución completa (referencia)

La siguiente celda resume el flujo completo de la clase en una versión compacta y ordenada.


In [25]:
# ============================================================
# Referencia compacta del pipeline completo
# ============================================================
df_modelo = df.copy()

def quitar_tildes(texto):
    return "".join(
        caracter
        for caracter in unicodedata.normalize("NFD", str(texto))
        if unicodedata.category(caracter) != "Mn"
    )

def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = quitar_tildes(texto)
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)
    texto = re.sub(r"@\w+", " ", texto)
    texto = re.sub(r"#\w+", " ", texto)
    texto = re.sub(r"[^a-z\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

stopwords_basicas = {
    "el", "la", "los", "las", "de", "del", "y", "o", "a", "en",
    "un", "una", "muy", "con", "mi", "me", "lo", "al", "por",
    "para", "que", "se", "sin", "fue", "es", "todo"
}

def tokenizar(texto):
    tokens = limpiar_texto(texto).split()
    return [t for t in tokens if t not in stopwords_basicas and len(t) > 1]

df_modelo["texto_limpio"] = df_modelo["texto"].apply(limpiar_texto)
df_modelo["tokens"] = df_modelo["texto"].apply(tokenizar)

X_train, X_test, y_train, y_test = train_test_split(
    df_modelo["texto"],
    df_modelo["sentimiento"],
    test_size=0.25,
    random_state=42,
    stratify=df_modelo["sentimiento"]
)

vectorizador = TfidfVectorizer(
    tokenizer=tokenizar,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=1
)

Xtr = vectorizador.fit_transform(X_train)
Xte = vectorizador.transform(X_test)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(Xtr, y_train)

pred = modelo.predict(Xte)

print("Accuracy final:", round(accuracy_score(y_test, pred), 3))
pd.DataFrame({
    "texto_test": X_test.values,
    "real": y_test.values,
    "prediccion": pred
}).head(10)


Accuracy final: 0.889


,texto_test,real,prediccion
0,VOLVERÍA A COMPRAR SIN PENSARLO EN SERIO #ENOJO,positivo,negativo
1,La comida llegó fría y sin sabor :) :(,negativo,negativo
2,La app falla y se cierra sola @soporte :(,negativo,negativo
3,"Esperaba mucho más, fue una mala experiencia #...",negativo,negativo
4,VOLVERÍA A COMPRAR SIN PENSARLO :) @SOPORTE,positivo,negativo
5,La experiencia fue excelente de principio a fi...,positivo,positivo
6,"No me gustó el producto, llegó dañado visita h...",negativo,negativo
7,EL CURSO ESTUVO CONFUSO Y MAL EXPLICADO POR FA...,negativo,negativo
8,Pésima calidad y demora en la entrega #enojo!!!,negativo,negativo
9,El soporte fue amable y resolvió todo 100% :(,positivo,positivo


---

# Cierre y takeaways

## Ideas clave que deben quedar claras

1. El texto **no entra directamente** a un modelo: primero hay que prepararlo.  
2. La limpieza y normalización ayudan a reducir ruido e inconsistencias.  
3. Tokenizar permite pasar del texto completo a unidades manejables.  
4. **BoW** cuenta términos; **TF-IDF** pondera su importancia.  
5. Con una buena representación numérica, ya es posible entrenar un clasificador sencillo.  
6. En problemas reales, la calidad del preprocesamiento y del dataset influye muchísimo en el resultado.

## Reto opcional para después de clase

Prueba alguna de estas extensiones:

- Cambiar `ngram_range=(1,2)` por `(1,3)` y comparar resultados.  
- Ampliar la lista de `stopwords_basicas`.  
- Probar otros textos inventados y revisar cómo predice el modelo.  
- Crear una tercera clase, por ejemplo `neutral`, y pensar qué cambiaría en el pipeline.  
- Investigar qué ventaja ofrecen los **embeddings** frente a TF-IDF.

## Mensaje final

Esta clase te deja una base sólida para entender el flujo mínimo de **Machine Learning para texto**.  
A partir de aquí ya puedes seguir hacia temas como:
- stemming y lematización real,
- embeddings,
- clasificación multiclase,
- análisis de temas,
- modelos más avanzados de NLP.
